# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 287.18it/s]


2026-05-24 12:51:01.489 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:907 - Data batch-empirical estimation of propensity score.


2026-05-24 12:51:01.497 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:956 - Data prediction of expected reward based on gbm model.


2026-05-24 12:51:02.870 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1073 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-05-24 12:51:33.003 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-05-24 12:51:33.399 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-05-24 12:51:33.797 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.477654,0.445365,0.510464,0.016694,b-ipw,reward_0
1,0.482961,0.482557,0.483367,0.000207,dm,reward_0
2,0.485417,0.452431,0.517996,0.016613,dr,reward_0
3,0.482961,0.482549,0.483369,0.000207,dros-opt,reward_0
4,0.485417,0.453691,0.518007,0.016593,dros-pess,reward_0
5,0.485007,0.452362,0.520288,0.017204,ipw,reward_0
6,0.486170,0.453580,0.518953,0.016757,rep,reward_0
7,0.485419,0.452621,0.518614,0.016655,sndr,reward_0
8,0.485346,0.451848,0.519742,0.017332,snips,reward_0
9,0.485417,0.452268,0.517139,0.016585,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 275.57it/s]


2026-05-24 12:51:34.353 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1305 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<08:30,  1.96it/s]

...

2026-05-24 12:52:20.864 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-05-24 12:52:21.263 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-05-24 12:52:21.660 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.483533,0.450965,0.514769,0.016239,b-ipw,reward_0
1,0.482912,0.482444,0.483382,0.000240,dm,reward_0
2,0.486398,0.454720,0.517906,0.016187,dr,reward_0
3,0.482912,0.482446,0.483382,0.000238,dros-opt,reward_0
4,0.486398,0.455863,0.517664,0.016076,dros-pess,reward_0
5,0.481135,0.448130,0.513115,0.016408,ipw,reward_0
6,0.487362,0.455392,0.520374,0.016746,rep,reward_0
7,0.486436,0.453863,0.518615,0.016427,sndr,reward_0
8,0.486450,0.454221,0.519462,0.016774,snips,reward_0
9,0.486398,0.454719,0.517642,0.016153,sg-dr,reward_0
